# Tyre degradation — grip(age) = 1 − a·age^b per compound (T5 groundwork)

The fitter never sees raw lap times. It works on the joint fit's *partial residuals*:
lap time with session, driver, compound, fuel and evolution removed and only the age
contribution added back, so the power law cannot re-absorb what E-10 already
explained. In/out laps and inaccurate laps are refused outright — an in-lap in a
degradation dataset is a cleaning failure wearing a physics costume (E-09).

A fit at the lower bound (a = 0) is a finding, not a failure: that circuit-era-compound
showed no measurable degradation signal, and the sample-size flags say how much weight
the claim can carry.

In [ ]:
import numpy as np
import pandas as pd
from senna import config
from senna.data.db import database_url, engine_for
from senna.physics.fits import frames, plots
from senna.physics.fits.gate import assert_quality_reports

fits_cfg = config.load("fits")
ingest_cfg = config.load("ingest")
CALIB = config.get(config.load("data_split"), "calibration_rounds")
SLICKS = ("SOFT", "MEDIUM", "HARD")

assert_quality_reports(
    config.get(ingest_cfg, "seasons"),
    config.REPO_ROOT / config.get(ingest_cfg, "quality.report_dir"),
)
engine = engine_for(database_url())
conn = await engine.connect()
index = await frames.circuit_index(conn, CALIB)
index[["slug", "era"]].drop_duplicates().groupby("era").size()

In [ ]:
excluded_rows = []
for era in sorted(index["era"].unique()):
    for slug in sorted(index.loc[index["era"] == era, "slug"].unique()):
        event_name = index.loc[index["slug"] == slug, "event_name"].iloc[0]
        laps = await frames.pace_laps(conn, event_name, era, CALIB)
        if laps.empty:
            continue
        in_out = (laps["pit_in_time_s"].notna() | laps["pit_out_time_s"].notna()).sum()
        excluded_rows.append(
            {
                "circuit": slug,
                "era": era,
                "rows": len(laps),
                "in_out_excluded": int(in_out),
                "inaccurate_excluded": int((~laps["is_accurate"].astype(bool)).sum()),
                "non_green_excluded": int((~laps["is_green"].astype(bool)).sum()),
            }
        )
excluded = pd.DataFrame(excluded_rows)
excluded.sum(numeric_only=True).to_frame("total").T

Per-fit exclusion counts — the E-09 reporting clause. Full table below.

In [ ]:
excluded

In [ ]:
from senna.physics.fits.degradation import fit_degradation
from senna.physics.fits.fuel import session_fuel
from senna.physics.fits.joint_pace import age_partial_residuals, fit_joint_pace

deg_cfg = config.get(fits_cfg, "degradation")
bounds = (
    [deg_cfg["curve_bounds"]["a"][0], deg_cfg["curve_bounds"]["b"][0]],
    [deg_cfg["curve_bounds"]["a"][1], deg_cfg["curve_bounds"]["b"][1]],
)
VIF_MAX = config.get(fits_cfg, "joint_pace.vif_max")
SC_BURN = config.get(fits_cfg, "fuel.sc_burn_factor")
fuel_by_era = {
    era: config.get(config.load(f"physics_{era}"), "mass.race_fuel_kg")
    for era in sorted(index["era"].unique())
}

deg_rows, residual_cache = [], {}
for era, allowance_kg in fuel_by_era.items():
    for slug in sorted(index.loc[index["era"] == era, "slug"].unique()):
        event_name = index.loc[index["slug"] == slug, "event_name"].iloc[0]
        laps = await frames.pace_laps(conn, event_name, era, CALIB)
        if laps.empty:
            continue
        laps = laps[laps["compound"].isin(SLICKS)]
        fuel_kg = session_fuel(laps, allowance_kg, SC_BURN)
        fit = fit_joint_pace(laps, fuel_kg, vif_max=VIF_MAX, calibration_rounds=CALIB)
        residuals = age_partial_residuals(laps, fuel_kg, calibration_rounds=CALIB)
        residual_cache[(slug, era)] = (residuals, fit.t0_s)
        for compound, deg in fit_degradation(
            residuals,
            fit.t0_s,
            bounds=bounds,
            min_laps=deg_cfg["min_laps"],
            min_stints=deg_cfg["min_stints"],
            min_drivers=deg_cfg["min_drivers"],
            min_max_age=deg_cfg["min_max_age"],
        ).items():
            deg_rows.append(
                {
                    "circuit": slug,
                    "era": era,
                    "compound": compound,
                    "a": deg.a,
                    "b": deg.b,
                    "n_laps": deg.n_laps,
                    "max_age_seen": deg.max_age_seen,
                    "low_confidence": deg.low_confidence,
                }
            )
deg_table = pd.DataFrame(deg_rows)
deg_table

In [ ]:
for slug, era in [("bahrain_grand_prix", "2024"), ("japanese_grand_prix", "2026")]:
    residuals, t0_s = residual_cache[(slug, era)]
    for compound in sorted(residuals["compound"].unique()):
        row = deg_table.query(
            "circuit == @slug and era == @era and compound == @compound"
        ).iloc[0]
        on_compound = residuals[residuals["compound"] == compound]
        fig = plots.degradation_curve(
            on_compound["tyre_life"],
            on_compound["pace_residual_s"],
            a=row["a"],
            b=row["b"],
            t0_s=t0_s,
            title=f"{slug} {era} {compound}",
        )
        print(plots.save_figure(fig, "degradation", f"{slug}_{era}_{compound}"))

Low-confidence fits stay in the yaml with their flags — T5's cluster transfer needs
the coefficients either way, and the flag travels with every consumer.

In [ ]:
await conn.close()
await engine.dispose()